# Trabajo de Laboratorio N°1:
## Efectos de Audio

### Setup & Herramientas

Para poder correr el código, deberán instalar las siguientes librerías de Python.

*Windows:*

In [1]:
pip install numpy scipy sounddevice matplotlib

Note: you may need to restart the kernel to use updated packages.


Importar librerías

In [2]:
import numpy as np
import scipy.io.wavfile as wav
import sounddevice as sd
from IPython.display import Audio, display

# Global settings
FS = 44100  # Sample rate in Hz

### Grabado de Audio

In [33]:
def record_audio(duration=5):
    print(f"Recording for {duration} seconds...")
    recording = sd.rec(int(duration * FS), samplerate=FS, channels=1)
    sd.wait()
    print("Recording finished.")
    # Flatten to 1D and normalize
    return recording.flatten().astype(np.float32)

# Cell to record and overwrite
input_audio = record_audio()
display(Audio(input_audio, rate=FS))

Recording for 5 seconds...
Recording finished.


### PARTE 1: Offline Processing

#### Efecto N°1: Eco

In [ ]:
# Effect: Echo
# Idea: Add a delayed version of the signal to itself, creating a repeating echo effect.
def apply_echo(input, delay_seconds, alpha, fs=44100):
    ########################################################
    delay_samples = delay_seconds * fs
    out = []
    for i in range(input.size):
        if(i>delay_samples):
          out.append(input[i] + alpha*input[int(i-delay_samples)])
        else:
          out.append(input[i])
    return out
# Processing Offline
delay_seconds = 0.5
alpha = 0.3
fs=44100
delay_samples = fs * delay_seconds
arraysize = input_audio.size
echoed = apply_echo(input_audio, delay_seconds, alpha, fs=FS)

print("Offline Results Ready:")
display(Audio(echoed, rate=FS))

Offline Results Ready:


In [6]:
# Effect: Multiple Ecos
# Idea: Simulate the effect of sound reflecting in a room by adding multiple delayed and attenuated copies of the signal.
def apply_multiple_echos(input, delays, decays, fs):
    ########################################################
    # Acá completar la función
    ########################################################
    pass

# Processing Offline
# Delays y decays variables a gusto
delays = [0.2, 0.4, 0.6]
decays = [0.5, 0.3, 0.1]

multiple_echos = apply_multiple_echos(input_audio, decays=decays, delays=delays, fs=FS)
display(Audio(multiple_echos, rate=FS))

ValueError: No audio data found. Expecting filename, url, or data.

#### Efecto N°2: Reverberación

In [7]:
# Effect: Reverb
# Idea: Add a delayed version of the output to the input signal, creating a coupling effect.
def apply_reverb(input, delay_sec, alpha, fs=FS):
    out = []
    delay_samples = delay_sec * fs
    for i in range(input.size):
        if i>delay_samples:
            out.append(input[i]+alpha*out[int(i-delay_samples)])
        else:
           out.append(input[i])
    return out

# Processing Offline
delay_seconds = 0.1
alpha = 0.7
reverbed = apply_reverb(input_audio, delay_seconds, alpha)


print("Offline Results Ready:")
display(Audio(reverbed, rate=FS))

Offline Results Ready:


#### Efecto N°3: Clipping

In [31]:
# Effect: Clipping
def apply_clipping(input, gain, threshold, fs=FS):
    out = []
    for i in range(input.size):
        value = input[i]*gain
        if (value>threshold):
            out.append(threshold)
        elif (value<-threshold):
            out.append(threshold)
        else:
            out.append(value)
    return out

# Processing Offline

# Pruben distintos valores
gain = 2
threshold = 0.1

clipping = apply_clipping(input_audio, gain, threshold, fs=FS)

print("Offline Results Ready:")
display(Audio(clipping, rate=FS))

Offline Results Ready:


#### Efecto N°4: Flanging

In [48]:
# Effect: Flanging
def apply_flanging(input, alpha, max_delay, fs=FS):

    A = max_delay*FS/10
    k0 = max_delay*FS - A
    flfo = 500
    out = []
    for i in range(input.size):
        current_delay = k0 + A*np.cos(2*np.pi*flfo*(i/fs))
        if(i>current_delay):
          out.append(input[i] + alpha*input[int(i-current_delay)])
        else:
          out.append(input[i])
    return out

# Processing Offline

# Pruben distintos valores
max_delay = 0.07
alpha = 0.7

flanging = apply_flanging(input_audio, alpha, max_delay, fs=FS)

print("Offline Results Ready:")
display(Audio(flanging, rate=FS))

Offline Results Ready:


### Procesamiento en tiempo real

#### Template procesamiento en tiempo real

Les dejamos un template para procesar en tiempo real.

"sd.Stream" lee por bloques el micrófono, corre un "callback" que procesa los datos obtenidos y reproduce por el parlante los datos de salida del "callback". La cantidad de datos de salida debe coincidir con la cantidad de datos de entrada.

In [ ]:
# ======================
# Parameters
# ======================
fs = 44100

# Prueben distintos valores de blocksize y luego evalúen a qué afecta
blocksize = 1024

In [ ]:
# ======================
# Audio Callback
# ======================

def audio_callback(indata, outdata, frames, time, status):
    x = indata[:, 0]          # mono input
    y = np.zeros_like(x)      # output block

    # Acá debería ir su procesamiento por bloques

    # Same output as input as example
    y = x

    outdata[:, 0] = y

Corran la siguiente celda para habilitar el micrófono, el procesamiento en tiempo real y el parlante

In [ ]:
# ======================
# Start Stream
# ======================

stream = sd.Stream(channels=1,
                   samplerate=fs,
                   blocksize=blocksize,
                   dtype='float32',
                   callback=audio_callback)

stream.start()

print("Stream started. Run stream.stop() to stop it.")

Corran la siguiente celda para pausar

In [ ]:
stream.stop()
stream.close()
print("Stream stopped.")

#### Efecto N°1: Echo

In [ ]:
delay_seconds = 0.5
alpha = 0.7
delay_samples = int(delay_seconds * fs) # Esto es la k de la H(s)
history_x = np.zeros(delay_samples) # Esto nos sirve para guardar muestras pasadas, y tiene tamaño k
tiempos_procesamiento = []

def echo_callback(indata, outdata, frames, time_info, status):
    global history_x, tiempos_procesamiento
    t_inicio = time.perf_counter() #esto es para el 6.c
    
    x = indata[:, 0]          # mono input
    
    #Probamos primero haciendo procesamiento sin bloques y "andaba" a medias. Partimos de esto:
    # y = x + alpha * prev_x
    # prev_x = x.copy()
    #Y notamos que el eco se quedaba corto, era distinto al offline. Esto era porque el delay (k en la H(s))
    #No podía acceder a valores anteriores al bloque actual. Para solucionar esto, necesitamos implementar un 
    #buffer que almacene muestras pasadas y así poder acceder a las posiciones k-esimas anteriores.
    
    
    delayed_x = history_x[:frames]
    y = x + alpha * delayed_x #Aplicamos la función transferencia
    history_x = np.concatenate((history_x[frames:], x)) #guardamos el buffer actual y descartamos lo viejo
    outdata[:, 0] = y

    t_fin = time.perf_counter()
    tiempo_ms = (t_fin - t_inicio) * 1000
    tiempos_procesamiento.append(tiempo_ms)

stream = sd.Stream(channels=1,
                   samplerate=fs,
                   blocksize=blocksize,
                   dtype='float32',
                   callback=echo_callback)

stream.start()
print("Stream started. Run stream.stop() to stop it.")

#Código parte 6.c :
#time.sleep(3) # Dejamos correr 3 segundos para juntar datos
#stream.stop()

#print(f"Latencia teórica por tamaño de bloque: {(blocksize/fs)*1000:.2f} ms")
#print(f"Tiempo de procesamiento promedio por bloque en Python: {np.mean(tiempos_procesamiento):.4f} ms")
#print(f"Peor caso de tiempo de procesamiento (): {np.max(tiempos_procesamiento):.4f} ms")

#Con estos prints vimos que el tiempo de procesamiento por Python era mucho menor
#a la que quedaba "hardcodeada" por el tamaño del bloque usado, que en el caso usado era 23,2 ms. Osea, el 
#procesamiento con python apenas cambia la latencia. Lo que más la determina es el tamaño del bloque.


#### Efecto N°2: Reverb